# 12 — Deploy Model Version 2 to Databricks Model Serving

This notebook updates the existing `worldbank-gep-intelligence-agent` endpoint to the newly registered Unity Catalog model Version 2.

Important:
- Uses the existing endpoint instead of creating a duplicate.
- Replaces the failed V1 served entity with V2.
- Keeps the SQL Warehouse ID as an environment variable.
- Does not hardcode a PAT/token.
- Stops if the deployment does not become ready.


In [0]:
# ============================================================
# CELL 1 — Install / verify Databricks SDK
# ============================================================
# Keep the SDK version consistent with the deployment API used
# by this notebook.

%pip install -q "databricks-sdk>=0.102.0"


In [0]:
# ============================================================
# CELL 2 — Restart Python after package installation
# ============================================================

dbutils.library.restartPython()


In [0]:
# ============================================================
# CELL 3 — Imports and production configuration
# ============================================================

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ServingModelWorkloadType,
)

MODEL_NAME = "worldbank_ai.ai.gep_intelligence_agent"

# Version 2 was successfully registered in Notebook 11.
MODEL_VERSION = "2"

# Reuse the existing endpoint. Do NOT create another endpoint name.
ENDPOINT_NAME = "worldbank-gep-intelligence-agent"

# Existing Databricks SQL Warehouse used by the structured Data Agent.
SQL_WAREHOUSE_ID = "3d11225dd32e8158"

print("Model:", MODEL_NAME)
print("Version:", MODEL_VERSION)
print("Endpoint:", ENDPOINT_NAME)
print("SQL Warehouse:", SQL_WAREHOUSE_ID)


In [0]:
# ============================================================
# CELL 4 — Initialize Databricks Workspace client
# ============================================================
# Uses Databricks-native notebook authentication.
# No PAT/token is hardcoded.

w = WorkspaceClient()

print("WorkspaceClient initialized successfully.")


In [0]:
# ============================================================
# CELL 5 — Confirm the existing endpoint
# ============================================================
# V1 previously created this endpoint but its deployment failed.
# We intentionally keep the same endpoint and update its config.

try:
    existing_endpoint = w.serving_endpoints.get(name=ENDPOINT_NAME)

    print("Existing endpoint found:", ENDPOINT_NAME)
    print("Current state:", existing_endpoint.state)

except Exception as exc:
    raise RuntimeError(
        f"Expected existing endpoint '{ENDPOINT_NAME}' was not found. "
        "Do not create a second endpoint until this is investigated."
    ) from exc


In [0]:
# ============================================================
# CELL 6 — Define the Version 2 served entity
# ============================================================
# Only V2 is supplied in the new served_entities configuration.
# This replaces the failed V1 served-model configuration.
#
# The model itself is an orchestration model, so CPU/Small is
# sufficient for the PyFunc container. The runtime then calls
# Databricks SQL, AI Search, and Foundation Model endpoints.

served_entity_v2 = ServedEntityInput(
    name="gep-intelligence-agent-v2",
    entity_name=MODEL_NAME,
    entity_version=MODEL_VERSION,
    workload_type=ServingModelWorkloadType.CPU,
    workload_size="Small",
    scale_to_zero_enabled=True,
    environment_vars={
        "DATABRICKS_SQL_WAREHOUSE_ID": SQL_WAREHOUSE_ID,
    },
)

print("V2 served entity configuration created.")
print("Served entity:", served_entity_v2.name)
print("Model version:", served_entity_v2.entity_version)


In [0]:
# ============================================================
# CELL 7 — Update the existing endpoint to Version 2
# ============================================================
# IMPORTANT:
# We use update_config_and_wait(), not create_and_wait(),
# because the endpoint already exists.
#
# Passing only served_entity_v2 means the endpoint's serving
# configuration is updated to the new model version rather than
# retaining the failed V1 served entity.

print("Updating existing endpoint to Model Version 2...")
print("This can take several minutes while the serving container starts.")

deployment = w.serving_endpoints.update_config_and_wait(
    name=ENDPOINT_NAME,
    served_entities=[
        served_entity_v2
    ],
)

print("\nEndpoint update call completed.")


In [0]:
# ============================================================
# DIAGNOSTIC — Inspect failed V2 served entity
# ============================================================
# This does NOT modify the endpoint.
# It only reads the failed deployment state.

endpoint = w.serving_endpoints.get(
    name=ENDPOINT_NAME
)

print("==========================================")
print("ENDPOINT STATE")
print("==========================================")
print(endpoint.state)

print("\n==========================================")
print("PENDING CONFIG")
print("==========================================")
print(endpoint.pending_config)

print("\n==========================================")
print("ACTIVE CONFIG")
print("==========================================")
print(endpoint.config)

print("\n==========================================")
print("SERVED ENTITY STATUS")
print("==========================================")

# V2 may be in pending_config because deployment failed
if endpoint.pending_config and endpoint.pending_config.served_entities:
    for entity in endpoint.pending_config.served_entities:
        print("\nServed entity:", entity.name)
        print("Model:", entity.entity_name)
        print("Version:", entity.entity_version)
        print("State:", entity.state)

In [0]:
# ============================================================
# DIAGNOSTIC — Get V2 model-serving build logs
# ============================================================
# IMPORTANT:
# served_model_name must match the served entity name from
# our V2 deployment configuration.

SERVED_ENTITY_NAME = "gep-intelligence-agent-v2"

try:
    logs = w.serving_endpoints.build_logs(
        name=ENDPOINT_NAME,
        served_model_name=SERVED_ENTITY_NAME,
    )

    print("==========================================")
    print("V2 BUILD LOGS")
    print("==========================================")

    # SDK response objects differ slightly by version,
    # so print the complete response first.
    print(logs)

except Exception as exc:
    print("Could not retrieve build logs using build_logs().")
    print("Error type:", type(exc).__name__)
    print("Error:", exc)

In [0]:
# ============================================================
# CELL 8 — Read and validate final deployment state
# ============================================================

endpoint = w.serving_endpoints.get(
    name=ENDPOINT_NAME
)

print("==========================================")
print("MODEL SERVING DEPLOYMENT STATUS")
print("==========================================")
print("Endpoint:", ENDPOINT_NAME)
print("Model:", MODEL_NAME)
print("Expected model version:", MODEL_VERSION)
print("Endpoint state:", endpoint.state)

# Display the served entities returned by Databricks.
if endpoint.config and endpoint.config.served_entities:
    print("\nActive served entities:")

    for entity in endpoint.config.served_entities:
        print(
            " -",
            entity.name,
            "| model:",
            entity.entity_name,
            "| version:",
            entity.entity_version,
        )
else:
    print("\nNo active served entities were returned in endpoint.config.")

# Fail loudly if Databricks did not report the endpoint as ready.
state_text = str(endpoint.state)

if "READY" not in state_text:
    raise RuntimeError(
        "Endpoint did not reach READY state. "
        "Stop here and inspect the deployment/build logs before "
        "running Notebook 13."
    )

print("\n==========================================")
print("MODEL VERSION 2 DEPLOYMENT READY")
print("==========================================")
print("Next notebook: 13_endpoint_smoke_tests")


## Stop condition

Only continue to `13_endpoint_smoke_tests` if Cell 8 reports the endpoint as READY and the active served entity is `gep-intelligence-agent-v2` using model version `2`.

If Cell 7 fails with `UPDATE_FAILED`, do not recreate the endpoint. Capture the exact error and serving build logs so the V2 container failure can be diagnosed directly.
